In [4]:
# ============================================================
# Mobile Product Segmentation and Recommendation System
# ============================================================

# Import required libraries

import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")

Libraries imported successfully!


In [5]:
# ============================================================
# STEP 1: LOAD RAW DATASET
# ============================================================

file_path = r"Input\Mobile Reviews Sentiment null.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (50000, 22)


In [6]:
df.head()

,review_id,customer_name,age,brand,model,price_usd,price_local,currency,exchange_rate_to_usd,rating,...,language,review_date,verified_purchase,battery_life_rating,camera_rating,performance_rating,design_rating,display_rating,helpful_votes,source
0,1,Aryan Maharaj,45,Realme,Realme 12 Pro,337.31,₹27996.73,INR,83.00,2.0,...,Hindi,11/6/2023,True,1,1,3,2,1,1,Amazon
1,2,Davi Miguel Sousa,18,Realme,Realme 12 Pro,307.78,R$1754.35,BRL,5.70,4.0,...,Portuguese,3/30/2023,True,3,2,4,3,2,5,Flipkart
2,3,Pahal Balay,27,Google,Pixel 6,864.53,₹71755.99,INR,83.00,4.0,...,Hindi,12/7/2022,True,3,5,3,2,4,8,AliExpress
3,4,David Guzman,19,Xiaomi,Redmi Note 13,660.94,د.إ2425.65,AED,3.67,3.0,...,English,3/11/2025,False,1,3,2,1,2,3,Amazon
4,5,Yago Leão,38,Motorola,Edge 50,792.13,R$4515.14,BRL,5.70,3.0,...,Portuguese,9/29/2023,True,3,3,2,2,1,0,BestBuy


In [7]:
# ============================================================
# DATASET INFORMATION
# ============================================================

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   review_id             50000 non-null  int64  
 1   customer_name         50000 non-null  str    
 2   age                   50000 non-null  int64  
 3   brand                 50000 non-null  str    
 4   model                 50000 non-null  str    
 5   price_usd             47550 non-null  float64
 6   price_local           47569 non-null  str    
 7   currency              50000 non-null  str    
 8   exchange_rate_to_usd  50000 non-null  float64
 9   rating                47547 non-null  float64
 10  sentiment             47555 non-null  str    
 11  country               50000 non-null  str    
 12  language              50000 non-null  str    
 13  review_date           50000 non-null  str    
 14  verified_purchase     50000 non-null  bool   
 15  battery_life_rating   50000 no

In [8]:
df.columns

Index(['review_id', 'customer_name', 'age', 'brand', 'model', 'price_usd',
       'price_local', 'currency', 'exchange_rate_to_usd', 'rating',
       'sentiment', 'country', 'language', 'review_date', 'verified_purchase',
       'battery_life_rating', 'camera_rating', 'performance_rating',
       'design_rating', 'display_rating', 'helpful_votes', 'source'],
      dtype='str')

In [9]:
df.select_dtypes(include="str").nunique()

customer_name    45911
brand                7
model               22
price_local      46127
currency             8
sentiment            3
country              8
language             4
review_date       1096
source               5
dtype: int64

In [10]:
df["brand"].value_counts()

brand
Xiaomi      7241
Google      7234
Apple       7144
OnePlus     7136
Realme      7132
Motorola    7061
Samsung     7052
Name: count, dtype: int64

In [11]:
df["model"].value_counts()

model
Realme Narzo 70    3597
Realme 12 Pro      3535
Redmi Note 13      2470
Pixel 7a           2468
Mi 13 Pro          2445
OnePlus 11R        2433
Razr 40            2395
Pixel 8            2387
OnePlus Nord 3     2380
Pixel 6            2379
Moto G Power       2351
Poco X6            2326
OnePlus 12         2323
Edge 50            2315
iPhone SE          1796
iPhone 15 Pro      1793
Galaxy Note 20     1789
iPhone 14          1788
Galaxy A55         1788
iPhone 13          1767
Galaxy S24         1761
Galaxy Z Flip      1714
Name: count, dtype: int64

In [12]:
# ============================================================
# DUPLICATE CHECK - COMPLETE ROWS
# ============================================================

duplicate_rows = df.duplicated().sum()

print("Total duplicate rows:", duplicate_rows)

Total duplicate rows: 0


In [13]:
# ============================================================
# DUPLICATE CHECK - REVIEW ID
# ============================================================

duplicate_review_ids = df["review_id"].duplicated().sum()

print("Total duplicate review IDs:", duplicate_review_ids)

Total duplicate review IDs: 0


In [14]:
df.loc[
    df["price_usd"].isna(),
    ["brand", "model", "price_usd", "price_local",
     "currency", "exchange_rate_to_usd"]
].head(10)

,brand,model,price_usd,price_local,currency,exchange_rate_to_usd
44,Xiaomi,Mi 13 Pro,NaN,$267.21,USD,1.00
51,Apple,iPhone 14,NaN,A$2198.11,AUD,1.53
87,Xiaomi,Mi 13 Pro,NaN,A$334.06,AUD,1.53
92,Motorola,Edge 50,NaN,€489.82,EUR,0.93
110,Google,Pixel 7a,NaN,£509.18,GBP,0.78
142,Motorola,Moto G Power,NaN,$503.76,USD,1.00
173,Apple,iPhone 13,NaN,C$1861.48,CAD,1.38
179,Realme,Realme Narzo 70,NaN,د.إ1213.3,AED,3.67
201,Realme,Realme Narzo 70,NaN,£203.91,GBP,0.78
255,Samsung,Galaxy Z Flip,NaN,A$1804.67,AUD,1.53


In [16]:
# Find price_local values that cannot be converted
# after removing non-numeric characters

test_price = (
    df.loc[df["price_usd"].isna(), "price_local"]
      .str.replace(r"[^0-9.]", "", regex=True)
)

invalid_price = test_price[
    pd.to_numeric(test_price, errors="coerce").isna()
]

print("Total invalid extracted price values:", len(invalid_price))

invalid_price.head(20)

Total invalid extracted price values: 389


179      .1213.3
269       .856.1
478     .1858.27
700     .2037.99
766      .4591.5
995     .1076.78
1209    .3857.43
1303     .1238.7
1428         NaN
1489    .2397.32
1494    .1968.62
1510     .956.03
1711    .2213.89
1898    .2143.65
1899    .2468.04
1910    .2825.53
1956    .1672.35
2048         NaN
2114    .1444.88
2146    .2445.72
Name: price_local, dtype: str

In [17]:
df.loc[invalid_price.index, ["price_local", "currency", "exchange_rate_to_usd"]].head(20)

,price_local,currency,exchange_rate_to_usd
179,د.إ1213.3,AED,3.67
269,د.إ856.1,AED,3.67
478,د.إ1858.27,AED,3.67
700,د.إ2037.99,AED,3.67
766,د.إ4591.5,AED,3.67
995,د.إ1076.78,AED,3.67
1209,د.إ3857.43,AED,3.67
1303,د.إ1238.7,AED,3.67
1428,NaN,GBP,0.78
1489,د.إ2397.32,AED,3.67


In [18]:
test_price = (
    df.loc[df["price_usd"].isna(), "price_local"]
      .str.extract(r"([0-9]+(?:\.[0-9]+)?)", expand=False)
)

print(test_price.head(20))

44       267.21
51      2198.11
87       334.06
92       489.82
110      509.18
142      503.76
173     1861.48
179      1213.3
201      203.91
255     1804.67
261      883.15
269       856.1
308    40650.08
339    58778.94
343      710.36
344     1031.99
379           1
407     1013.27
471    46235.98
478     1858.27
Name: price_local, dtype: str


In [19]:
price_usd_calculated = (
    test_price.astype(float)
    / df.loc[df["price_usd"].isna(), "exchange_rate_to_usd"]
)

print(price_usd_calculated.head(20))

44      267.210000
51     1436.673203
87      218.339869
92      526.688172
110     652.794872
142     503.760000
173    1348.898551
179     330.599455
201     261.423077
255    1179.522876
261    1132.243590
269     233.269755
308     489.760000
339     708.180000
343     464.287582
344     747.818841
379       1.000000
407     662.267974
471     557.060000
478     506.340599
dtype: float64


In [20]:
id="q8v2ks"
recoverable_price = (
    df["price_usd"].isna()
    & df["price_local"].notna()
    & df["exchange_rate_to_usd"].notna()
)

print("Missing price_usd:", df["price_usd"].isna().sum())
print("Recoverable price_usd:", recoverable_price.sum())
print("Not recoverable:", (~recoverable_price & df["price_usd"].isna()).sum())

Missing price_usd: 2450
Recoverable price_usd: 2327
Not recoverable: 123


In [21]:
unrecoverable_price = df.loc[
    df["price_usd"].isna() & ~recoverable_price,
    ["price_usd", "price_local", "currency", "exchange_rate_to_usd", "brand", "model"]
]

print("Unrecoverable rows:", len(unrecoverable_price))

unrecoverable_price.head(20)

Unrecoverable rows: 123


,price_usd,price_local,currency,exchange_rate_to_usd,brand,model
1428,NaN,NaN,GBP,0.78,Xiaomi,Poco X6
2048,NaN,NaN,AUD,1.53,OnePlus,OnePlus 11R
2193,NaN,NaN,USD,1.00,OnePlus,OnePlus 11R
2310,NaN,NaN,BRL,5.70,Xiaomi,Poco X6
2774,NaN,NaN,INR,83.00,OnePlus,OnePlus 12
2962,NaN,NaN,AUD,1.53,Samsung,Galaxy A55
3377,NaN,NaN,AUD,1.53,Xiaomi,Redmi Note 13
3696,NaN,NaN,AED,3.67,Motorola,Razr 40
3870,NaN,NaN,AED,3.67,Samsung,Galaxy S24
4482,NaN,NaN,AUD,1.53,Apple,iPhone 15 Pro


In [22]:
# ============================================================
# Recover price_usd using price_local and exchange_rate_to_usd
# ============================================================

price_local_numeric = (
    df["price_local"]
      .str.extract(r"([0-9]+(?:\.[0-9]+)?)", expand=False)
)

recoverable_price = (
    df["price_usd"].isna()
    & price_local_numeric.notna()
)

df.loc[recoverable_price, "price_usd"] = (
    price_local_numeric[recoverable_price].astype(float)
    / df.loc[recoverable_price, "exchange_rate_to_usd"]
)

print("Remaining missing price_usd:", df["price_usd"].isna().sum())

Remaining missing price_usd: 123


In [23]:
# ============================================================
# Fill remaining price_usd using model-wise median
# ============================================================

df["price_usd"] = (
    df.groupby("model")["price_usd"]
      .transform(lambda x: x.fillna(x.median()))
)

print("Remaining missing price_usd:", df["price_usd"].isna().sum())

Remaining missing price_usd: 0


In [24]:
print("Missing:", df["price_usd"].isna().sum())
print("Minimum:", df["price_usd"].min())
print("Maximum:", df["price_usd"].max())
print("Data type:", df["price_usd"].dtype)

Missing: 0
Minimum: 1.0
Maximum: 1499.89
Data type: float64


In [25]:
df.loc[
    df["price_usd"] == df["price_usd"].min(),
    ["review_id", "brand", "model", "price_usd", "price_local",
     "currency", "exchange_rate_to_usd", "rating"]
]

,review_id,brand,model,price_usd,price_local,currency,exchange_rate_to_usd,rating
379,380,Apple,iPhone 13,1.0,"$1,468.29",USD,1.0,5.0
542,543,Google,Pixel 7a,1.0,"$1,007.72",USD,1.0,3.0
1807,1808,Apple,iPhone 15 Pro,1.0,"$1,318.46",USD,1.0,3.0
2247,2248,Apple,iPhone 14,1.0,"$1,294.86",USD,1.0,NaN
3168,3169,Apple,iPhone 15 Pro,1.0,"$1,151.36",USD,1.0,5.0
4404,4405,Samsung,Galaxy S24,1.0,"$1,156.56",USD,1.0,4.0
4545,4546,Apple,iPhone 14,1.0,"$1,014.05",USD,1.0,1.0
6049,6050,Google,Pixel 7a,1.0,"$1,076.52",USD,1.0,5.0
7010,7011,Samsung,Galaxy Z Flip,1.0,"$1,240.16",USD,1.0,3.0
7365,7366,Google,Pixel 7a,1.0,"$1,043.81",USD,1.0,4.0


In [26]:
print("Rows with price_usd = 1:", (df["price_usd"] == 1).sum())

Rows with price_usd = 1: 47


In [27]:
price_one_calculated = (
    df.loc[df["price_usd"] == 1, "price_local"]
      .str.extract(r"([0-9]+(?:\.[0-9]+)?)", expand=False)
      .astype(float)
    / df.loc[df["price_usd"] == 1, "exchange_rate_to_usd"]
)

print(price_one_calculated.head(10))

379     1.0
542     1.0
1807    1.0
2247    1.0
3168    1.0
4404    1.0
4545    1.0
6049    1.0
7010    1.0
7365    1.0
dtype: float64


In [28]:
test_one = (
    df.loc[df["price_usd"] == 1, "price_local"]
      .str.replace(",", "", regex=False)
      .str.extract(r"([0-9]+(?:\.[0-9]+)?)", expand=False)
)

print(test_one.head(10))

379     1468.29
542     1007.72
1807    1318.46
2247    1294.86
3168    1151.36
4404    1156.56
4545    1014.05
6049    1076.52
7010    1240.16
7365    1043.81
Name: price_local, dtype: str


In [29]:
price_one_corrected = (
    test_one.astype(float)
    / df.loc[df["price_usd"] == 1, "exchange_rate_to_usd"]
)

print(price_one_corrected.head(10))

379     1468.29
542     1007.72
1807    1318.46
2247    1294.86
3168    1151.36
4404    1156.56
4545    1014.05
6049    1076.52
7010    1240.16
7365    1043.81
dtype: float64


In [30]:
df.loc[df["price_usd"] == 1, "price_usd"] = price_one_corrected

print("Rows with price_usd = 1:", (df["price_usd"] == 1).sum())
print("Missing price_usd:", df["price_usd"].isna().sum())

Rows with price_usd = 1: 0
Missing price_usd: 0


In [31]:
print(df["price_usd"].describe())

count    50000.000000
mean       689.693230
std        310.098186
min        180.020000
25%        451.040000
50%        637.125000
75%        900.917500
max       1499.890000
Name: price_usd, dtype: float64


In [32]:
df.isna().sum()

review_id                  0
customer_name              0
age                        0
brand                      0
model                      0
price_usd                  0
price_local             2431
currency                   0
exchange_rate_to_usd       0
rating                  2453
sentiment               2445
country                    0
language                   0
review_date                0
verified_purchase          0
battery_life_rating        0
camera_rating              0
performance_rating         0
design_rating              0
display_rating             0
helpful_votes              0
source                  2448
dtype: int64

In [33]:
df["rating"].describe()

count    47547.000000
mean         3.123415
std          1.249067
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max          5.000000
Name: rating, dtype: float64

In [34]:
pd.crosstab(
    df["sentiment"],
    df["rating"],
    normalize="index"
).round(3)

rating,1.0,2.0,3.0,4.0,5.0
sentiment,,,,,
Negative,0.503,0.391,0.099,0.006,0.000
Neutral,0.109,0.406,0.379,0.099,0.007
Positive,0.001,0.030,0.244,0.461,0.264


In [35]:
missing_rating = df["rating"].isna()

print("Missing rating:", missing_rating.sum())
print("Missing rating + sentiment available:",
      df.loc[missing_rating, "sentiment"].notna().sum())
print("Missing rating + sentiment missing:",
      df.loc[missing_rating, "sentiment"].isna().sum())

Missing rating: 2453
Missing rating + sentiment available: 2326
Missing rating + sentiment missing: 127


In [36]:
df.groupby("sentiment")["rating"].agg(["count", "mean", "median"]).round(2)

,count,mean,median
sentiment,,,
Negative,8862,1.61,1.0
Neutral,11349,2.49,2.0
Positive,25018,3.96,4.0


In [37]:
df.loc[df["rating"].isna(), "sentiment"].value_counts(dropna=False)

sentiment
Positive    1264
Neutral      582
Negative     480
NaN          127
Name: count, dtype: int64

In [38]:
rating_columns = [
    "rating",
    "battery_life_rating",
    "camera_rating",
    "performance_rating",
    "design_rating",
    "display_rating"
]

df[rating_columns].corr()["rating"].sort_values(ascending=False)

rating                 1.000000
battery_life_rating    0.762147
camera_rating          0.760083
display_rating         0.757411
performance_rating     0.755611
design_rating          0.755272
Name: rating, dtype: float64

In [39]:
missing_both = df["rating"].isna() & df["sentiment"].isna()

df.loc[
    missing_both,
    [
        "model",
        "age",
        "battery_life_rating",
        "camera_rating",
        "performance_rating",
        "design_rating",
        "display_rating"
    ]
].isna().sum()

model                  0
age                    0
battery_life_rating    0
camera_rating          0
performance_rating     0
design_rating          0
display_rating         0
dtype: int64

In [40]:
component_avg = df[
    [
        "battery_life_rating",
        "camera_rating",
        "performance_rating",
        "design_rating",
        "display_rating"
    ]
].mean(axis=1)

comparison = pd.DataFrame({
    "actual_rating": df["rating"],
    "component_avg": component_avg
})

comparison.corr().round(3)

,actual_rating,component_avg
actual_rating,1.000,0.929
component_avg,0.929,1.000


In [41]:
df.loc[
    missing_both,
    [
        "model",
        "battery_life_rating",
        "camera_rating",
        "performance_rating",
        "design_rating",
        "display_rating"
    ]
].assign(
    component_avg=lambda x: x[
        [
            "battery_life_rating",
            "camera_rating",
            "performance_rating",
            "design_rating",
            "display_rating"
        ]
    ].mean(axis=1)
)["component_avg"].describe()

count    127.000000
mean       2.639370
std        1.091683
min        1.000000
25%        1.600000
50%        2.600000
75%        3.600000
max        4.800000
Name: component_avg, dtype: float64

In [42]:
# Impute missing ratings using sentiment

sentiment_rating_map = {
    "Negative": 1,
    "Neutral": 2,
    "Positive": 4
}

missing_rating_with_sentiment = (
    df["rating"].isna() & df["sentiment"].notna()
)

df.loc[missing_rating_with_sentiment, "rating"] = (
    df.loc[missing_rating_with_sentiment, "sentiment"]
      .map(sentiment_rating_map)
)

print("Remaining missing rating:", df["rating"].isna().sum())

Remaining missing rating: 127


In [43]:
missing_both = df["rating"].isna() & df["sentiment"].isna()

component_avg_missing = df.loc[
    missing_both,
    [
        "battery_life_rating",
        "camera_rating",
        "performance_rating",
        "design_rating",
        "display_rating"
    ]
].mean(axis=1)

print(component_avg_missing.head(10))

69      2.0
259     3.0
791     1.4
1360    2.2
1377    3.4
1514    1.8
1620    4.8
1677    3.8
1950    1.6
2173    1.8
dtype: float64


In [44]:
# Round component average to the nearest valid rating (1–5)

df.loc[missing_both, "rating"] = (
    component_avg_missing
    .round()
    .clip(1, 5)
)

print("Remaining missing rating:", df["rating"].isna().sum())

Remaining missing rating: 0


In [45]:
print(df["rating"].describe())
print("\nRating distribution:")
print(df["rating"].value_counts().sort_index())

count    50000.000000
mean         3.110780
std          1.250959
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max          5.000000
Name: rating, dtype: float64

Rating distribution:
rating
1.0     6578
2.0     9921
3.0    11889
4.0    14608
5.0     7004
Name: count, dtype: int64


In [46]:
pd.crosstab(
    df["rating"],
    df["sentiment"],
    normalize="index"
).round(3)

sentiment,Negative,Neutral,Positive
rating,,,
1.0,0.796,0.200,0.004
2.0,0.368,0.552,0.080
3.0,0.078,0.381,0.540
4.0,0.004,0.080,0.916
5.0,0.000,0.011,0.989


In [47]:
df.loc[df["sentiment"].isna(), "rating"].value_counts().sort_index()

rating
1.0    370
2.0    506
3.0    614
4.0    641
5.0    314
Name: count, dtype: int64

In [48]:
# Impute missing sentiment using the validated rating-sentiment relationship

rating_sentiment_map = {
    1.0: "Negative",
    2.0: "Neutral",
    3.0: "Positive",
    4.0: "Positive",
    5.0: "Positive"
}

missing_sentiment = df["sentiment"].isna()

df.loc[missing_sentiment, "sentiment"] = (
    df.loc[missing_sentiment, "rating"]
      .map(rating_sentiment_map)
)

print("Remaining missing sentiment:", df["sentiment"].isna().sum())

Remaining missing sentiment: 0


In [49]:
df["source"].value_counts(dropna=False)

source
eBay          9573
AliExpress    9531
Amazon        9513
Flipkart      9481
BestBuy       9454
NaN           2448
Name: count, dtype: int64

In [50]:
pd.crosstab(
    df["brand"],
    df["source"],
    normalize="index"
).round(3)

source,AliExpress,Amazon,BestBuy,Flipkart,eBay
brand,,,,,
Apple,0.202,0.198,0.195,0.208,0.196
Google,0.202,0.208,0.193,0.192,0.205
Motorola,0.201,0.202,0.198,0.195,0.204
OnePlus,0.202,0.196,0.203,0.200,0.199
Realme,0.194,0.199,0.198,0.202,0.207
Samsung,0.200,0.198,0.198,0.200,0.203
Xiaomi,0.202,0.199,0.207,0.198,0.194


In [51]:
pd.crosstab(
    df["country"],
    df["source"],
    normalize="index"
).round(3)

source,AliExpress,Amazon,BestBuy,Flipkart,eBay
country,,,,,
Australia,0.186,0.197,0.192,0.213,0.211
Brazil,0.206,0.199,0.197,0.205,0.193
Canada,0.204,0.197,0.207,0.192,0.200
Germany,0.205,0.192,0.196,0.195,0.211
India,0.199,0.197,0.200,0.205,0.199
UAE,0.198,0.206,0.202,0.186,0.208
UK,0.201,0.204,0.198,0.203,0.194
USA,0.206,0.208,0.197,0.196,0.194


In [52]:
group_mode_check = (
    df.groupby(["brand", "country"])["source"]
      .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown")
      .reset_index()
)

print(group_mode_check)

       brand    country      source
0      Apple  Australia    Flipkart
1      Apple     Brazil  AliExpress
2      Apple     Canada        eBay
3      Apple    Germany     BestBuy
4      Apple      India    Flipkart
5      Apple        UAE  AliExpress
6      Apple         UK      Amazon
7      Apple        USA        eBay
8     Google  Australia        eBay
9     Google     Brazil      Amazon
10    Google     Canada  AliExpress
11    Google    Germany        eBay
12    Google      India      Amazon
13    Google        UAE      Amazon
14    Google         UK        eBay
15    Google        USA      Amazon
16  Motorola  Australia    Flipkart
17  Motorola     Brazil        eBay
18  Motorola     Canada  AliExpress
19  Motorola    Germany  AliExpress
20  Motorola      India      Amazon
21  Motorola        UAE     BestBuy
22  Motorola         UK  AliExpress
23  Motorola        USA     BestBuy
24   OnePlus  Australia      Amazon
25   OnePlus     Brazil     BestBuy
26   OnePlus     Canada     

In [53]:
# Impute missing source using Brand + Country group mode

df["source"] = df.groupby(
    ["brand", "country"]
)["source"].transform(
    lambda x: x.fillna(x.mode().iloc[0])
)

print("Missing source values:", df["source"].isna().sum())
print("\nSource distribution after imputation:")
print(df["source"].value_counts())

Missing source values: 0

Source distribution after imputation:
source
eBay          10234
BestBuy       10033
Amazon         9967
AliExpress     9934
Flipkart       9832
Name: count, dtype: int64


In [54]:
print("Missing values after cleaning:")
print(df.isnull().sum())

Missing values after cleaning:
review_id                  0
customer_name              0
age                        0
brand                      0
model                      0
price_usd                  0
price_local             2431
currency                   0
exchange_rate_to_usd       0
rating                     0
sentiment                  0
country                    0
language                   0
review_date                0
verified_purchase          0
battery_life_rating        0
camera_rating              0
performance_rating         0
design_rating              0
display_rating             0
helpful_votes              0
source                     0
dtype: int64


In [55]:
# Convert review_date from text to datetime
df["review_date"] = pd.to_datetime(
    df["review_date"],
    errors="coerce"
)

print("Data type:", df["review_date"].dtype)
print("Missing dates:", df["review_date"].isna().sum())

Data type: datetime64[us]
Missing dates: 0


In [56]:
print("Date range:")
print("Minimum:", df["review_date"].min().strftime("%m/%d/%Y"))
print("Maximum:", df["review_date"].max().strftime("%m/%d/%Y"))

print("\nSample dates in US format:")
print(df["review_date"].head().dt.strftime("%m/%d/%Y"))

Date range:
Minimum: 10/22/2022
Maximum: 10/21/2025

Sample dates in US format:
0    11/06/2023
1    03/30/2023
2    12/07/2022
3    03/11/2025
4    09/29/2023
Name: review_date, dtype: str


In [57]:
print(df["review_date"].head())
print("\nData type:", df["review_date"].dtype)

0   2023-11-06
1   2023-03-30
2   2022-12-07
3   2025-03-11
4   2023-09-29
Name: review_date, dtype: datetime64[us]

Data type: datetime64[us]


In [58]:
# Save cleaned dataset
output_path = r"Output\Mobile_Reviews_Cleaned.csv"

df.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")
print("File:", output_path)
print("Shape:", df.shape)

Cleaned dataset saved successfully!
File: Output\Mobile_Reviews_Cleaned.csv
Shape: (50000, 22)
